# Who gets a fuel estimate?

This notebook explains the **exploratory** batch eligibility audit. Run from the repository root after installing `requirements.txt`. The source of truth is `src/flight_fuel/eligibility.py`; this notebook reads its committed result and creates presentation tables and figures. It does not retrain a model. October had already been examined when this rule was developed; the revised base fit also disables the historical row-random internal early stopping.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

report = json.loads(Path("reports/eligibility-results.json").read_text())
report["status"]


## What is the label and who can benefit?

Each row is a completed ACARS-defined interval. `fuel_kg` is a difference between fuel reports. A researcher or data curator can use this audit to decide whether a public-data sample covers the durations and aircraft types required for a comparison. Neither the selected-case error nor the group floor guarantees individual accuracy.


In [ ]:
rows = []
for period, data in report["periods"].items():
    for policy, p in data["policies"].items():
        rows.append({"period": period, "policy": policy, "intervals": p["n"],
                     "accepted_%": 100*p["coverage"],
                     "accepted_MAE_kg": p["accepted_mae_kg"],
                     "rejected_MAE_kg": p["rejected_mae_kg"],
                     "labeled_fuel_represented_%": 100*p["accepted_reference_fuel_fraction"]})
pd.DataFrame(rows).round(2)


## Why the access floor matters

The original risk selector tended to reject almost every longer interval. Compare the new policies on **the same underlying fuel predictions**. Better group access can raise accepted-case error; rejected intervals remain in the denominator when we discuss sample coverage. Fuel share uses labels *after* selection for evaluation only.


In [ ]:
period = report["periods"]["validation"]["policies"]
groups = ["<5 min", "5–15 min", "15–60 min"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for policy, style in [("risk_only", "#ad6437"), ("representation_floor", "#117f77")]:
    data = period[policy]["groups"]["duration"]
    axes[0].plot(groups, [100*data[g]["coverage"] for g in groups], marker="o", label=policy, color=style)
    axes[1].plot(groups, [data[g]["accepted_mae_kg"] for g in groups], marker="o", label=policy, color=style)
axes[0].set(ylabel="Accepted intervals (%)", ylim=(0, 105), title="Access by duration")
axes[1].set(ylabel="MAE on accepted intervals (kg)", title="Error on different accepted sets")
for ax in axes: ax.tick_params(axis="x", rotation=15); ax.grid(alpha=.2)
axes[0].legend()
plt.show()


## Error distribution and route shift

The full-batch error distribution matters even when selective MAE is low. R² summarizes variance explained within this sample, not the accuracy of a particular flight. MAPE is diagnostic only because short intervals can have small positive fuel labels. Risk deciles check ordering empirically; they are not calibrated uncertainty bands. Route comparisons are confounded by the mix of aircraft and durations.


In [ ]:
d = report["periods"]["validation"]["residual_diagnostics"]
pd.DataFrame([d["full"]]).T.rename(columns={0: "September full batch"}).round(3)


In [ ]:
deciles = pd.DataFrame(d["risk_deciles"])
ax = deciles.plot(y=["mean_risk_score_kg", "observed_mae_kg"], marker="o", figsize=(7, 4))
ax.set(xlabel="Risk decile (lowest to highest)", ylabel="Mean absolute error (kg)", title="Risk ranking on September validation")
ax.set_xticks(range(10), [str(i) for i in range(1, 11)])
ax.grid(alpha=.2)
plt.show()


In [ ]:
pd.DataFrame(report["route_diagnostics"]["periods"]).T.round(2)


## Defending the result

- The fuel predictor uses type, duration and ADS-B summaries; the fallback uses type and duration on zero-ADS-B intervals. Fuel-report values never enter the prediction features.
- The risk model ranks expected absolute errors; its score is not a prediction interval. The access floor is a deterministic rule on batch scores, duration and aircraft group, not on reference fuel.
- The observed fuel share is only for this labeled benchmark sample. It is not a fleet-level emissions statistic.
- October scores for this *new* policy are exploratory because the earlier October weakness motivated the rule. An independently labeled later period is needed for confirmation.

- The revised base model explicitly disables the historical row-random internal holdout; this changes its fit, so the old Phase 4 score is not a controlled comparison.
For dataset provenance, exact grouping rules, and reproduction commands, see `docs/methods.md` and `README.md`.
